# Data Exploration of the Telco Churn Dataset

## Imports

In [6]:
import pandas as pd
pd.__version__

'3.0.5'

## Load Dataset 

In [3]:
df = pd.read_csv("../data/Telco_Churn.csv")
df.shape

(7043, 21)

In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

Total Charges shows non-null, but we need to check whether total charges is actually null all the way through

In [ ]:
df["TotalCharges"].str.strip().eq("").sum()

np.int64(11)

Above shows there are 11 customers with 0 "TotalCharges" which implies they are new customers with 0 tenure, we can check this with the below

In [ ]:
df.loc[df["TotalCharges"].str.strip().eq(""), ["tenure", "MonthlyCharges", "Churn"]]

,tenure,MonthlyCharges,Churn
488,0,52.55,No
753,0,20.25,No
936,0,80.85,No
1082,0,25.75,No
1340,0,56.05,No
3331,0,19.85,No
3826,0,25.35,No
4380,0,20.00,No
5218,0,19.70,No
6670,0,73.35,No


We need to convert the "TotalCharges" column to numeric, while avoiding errors with "coerce". 
In the same swing we want to convert any NaN values to 0 as we confirmed these are customers with "tenure=0"

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0) 

# Check the change worked

df["TotalCharges"].dtype

dtype('float64')

Let's run value counts for "Churn", both standard and normalised to see what values we have got.
Wrapping the non-normalised in print as it doesn't have an output by default.

In [14]:
print(df["Churn"].value_counts())
df["Churn"].value_counts(normalize=True)

Churn
No     5174
Yes    1869
Name: count, dtype: int64


Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64

Let's do a quick check for any duplicates in the customerID, this is something we don't want as it can skew the model.

In [15]:
df["customerID"].duplicated().sum()

np.int64(0)

Now let's check the churn rate on the data using the pd.crosstab tool.
This will show the percentage "Churn" per the type of contract. Rounded to 4 decimals for ease of reading.

In [17]:
pd.crosstab(df["Contract"], df["Churn"], normalize="index").round(4)

Churn,No,Yes
Contract,,
Month-to-month,0.5729,0.4271
One year,0.8873,0.1127
Two year,0.9717,0.0283
